
#### **Trabalho 1 - Aprendizado de Máquina - Decision Tree - Base de dados Diabetes**

Sumário:

---


1) **Descrição da base e variáveis (entendimento dos dados)**
2) **Estratégias de Pré-processamento, Imputação e Balanceamento dos Dados**
3) **Análise Descritiva dos dados**
4) **Preparação dos dados para modelagem**
5) **Treinamento e ajuste de hiperparâmetros**
6) **Avaliação e validação do modelo**
7) **Interpretação e análise dos resultados**
8) **Conclusão**


### **1) Descrição da base e variáveis (entendimento dos dados)**

### **Banco de dados de diabetes dos índios Pima**

**Contexto:** Este conjunto de dados é originário do Instituto Nacional de Diabetes e Doenças Digestivas e Renais. O objetivo do conjunto de dados é prever, de forma diagnóstica, se um paciente tem ou não diabetes, com base em determinadas medidas diagnósticas incluídas no conjunto de dados. Diversas restrições foram impostas à seleção dessas instâncias de um banco de dados maior. Em particular, todos os pacientes aqui são mulheres com pelo menos 21 anos de idade e de ascendência indígena Pima.


 **Dicionário de Variáveis — Pima Indians Diabetes Database**

| Variável | Descrição | Unidade / Tipo | Observações |
|-----------|------------|----------------|--------------|
| **Pregnancies** | Número de vezes que a paciente engravidou | Numérica (inteiro) | Pode variar de 0 a 17 |
| **Glucose** | Concentração plasmática de glicose em 2 horas em um teste oral de tolerância à glicose | mg/dL | Valor 0 indica dado ausente |
| **BloodPressure** | Pressão arterial diastólica | mm Hg | Valor 0 indica dado ausente |
| **SkinThickness** | Espessura da prega cutânea do tríceps | mm | Valor 0 indica dado ausente |
| **Insulin** | Insulina sérica de 2 h | μU/mL | Valor 0 indica dado ausente |
| **BMI** | Índice de massa corporal (peso / altura²) | kg/m² | Valor 0 indica dado ausente |
| **DiabetesPedigreeFunction** | Função de pedigree de diabetes (histórico familiar) | Adimensional | Mede predisposição genética |
| **Age** | Idade da paciente | Anos | Mínimo: 21  Máximo: 81 |
| **Outcome** | Resultado do teste de diagnóstico | Binária (0 = não diabética; 1 = diabética) | 268 de 768 registros são positivos |


In [ ]:
# ========================= 0. IMPORTS & CONFIG =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

# Reprodutibilidade
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
from google.colab import files

# Abre o seletor para escolher o arquivo CSV do seu computador
#uploaded = files.upload()


# Caminho do CSV - Trabalho 2 (Random Tree)
from google.colab import drive
drive.mount('/content/drive')
# Caminho completo até o CSV dentro da pasta
file_path = '/content/drive/MyDrive/Trabalho 1 - Aprendizado de Máquina /diabetes.csv'

# Replace with the correct path to your file
#file_path = '/path/to/your/diabetes.csv'
df = pd.read_csv(file_path)

# Carregar dataset
df = pd.read_csv(file_path)

### **2. Estratégias de Pré-processamento, Imputação e Balanceamento dos Dados**


In [ ]:
# Colunas com zeros fisiologicamente impossíveis
COLS_ZERO = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
TARGET = 'Outcome'
print("Dimensões da base:", df.shape)


In [ ]:
# ============================================================
#   CONSTRUÇÃO DAS BASES — TRABALHO 2 (RANDOM TREE)
#   (Apenas 5 modelos com MICE implementado nos modelos 4 e 5)
# ============================================================

# Variáveis globais utilizadas
TARGET = 'Outcome'
COLS_ZERO = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from imblearn.over_sampling import SMOTE

# ------------------------------------------------------------
# Base 1 — Removendo Zeros (Baseline)
# ------------------------------------------------------------
def base_2_removendo_zeros(dfin: pd.DataFrame) -> pd.DataFrame:
    """Remove registros que contêm zeros em variáveis críticas."""
    mask = (dfin[COLS_ZERO] != 0).all(axis=1)
    return dfin.loc[mask].copy()


# ------------------------------------------------------------
# Base 2 — MICE + SMOTE (Sem variável Insulin)
# ------------------------------------------------------------
def base_8_mice_smote_sem_insulin(dfin: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """Imputação MICE, balanceamento SMOTE e exclusão da variável Insulin."""
    dfc = dfin.copy()
    dfc[COLS_ZERO] = dfc[COLS_ZERO].replace(0, np.nan)

    # Imputação MICE
    imp = IterativeImputer(random_state=random_state, max_iter=10)
    dfc[COLS_ZERO] = imp.fit_transform(dfc[COLS_ZERO])

    # Balanceamento SMOTE
    X = dfc.drop(columns=[TARGET])
    y = dfc[TARGET]
    smote = SMOTE(random_state=random_state)
    X_res, y_res = smote.fit_resample(X, y)

    df_bal = pd.DataFrame(X_res, columns=X.columns)
    df_bal[TARGET] = y_res

    # Remove Insulin após o balanceamento
    if 'Insulin' in df_bal.columns:
        df_bal = df_bal.drop(columns=['Insulin'])
    return df_bal


# ------------------------------------------------------------
# Base 3 — MICE + SMOTE (Completo)
# ------------------------------------------------------------
def base_4_mice_smote(dfin: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """Imputação MICE seguida de balanceamento com SMOTE."""
    dfc = dfin.copy()
    dfc[COLS_ZERO] = dfc[COLS_ZERO].replace(0, np.nan)

    imp = IterativeImputer(random_state=random_state, max_iter=10)
    dfc[COLS_ZERO] = imp.fit_transform(dfc[COLS_ZERO])

    X = dfc.drop(columns=[TARGET])
    y = dfc[TARGET]
    smote = SMOTE(random_state=random_state)
    X_res, y_res = smote.fit_resample(X, y)

    df_bal = pd.DataFrame(X_res, columns=X.columns)
    df_bal[TARGET] = y_res
    return df_bal


# ------------------------------------------------------------
# Base 4 — Removendo Variáveis de Baixa Importância (com MICE)
# ------------------------------------------------------------
def base_5_removendo_baixa_importancia(dfin: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """Imputação MICE + remoção das variáveis SkinThickness e DiabetesPedigreeFunction."""
    dfc = dfin.copy()
    dfc[COLS_ZERO] = dfc[COLS_ZERO].replace(0, np.nan)

    # Imputação MICE
    imp = IterativeImputer(random_state=random_state, max_iter=10)
    dfc[COLS_ZERO] = imp.fit_transform(dfc[COLS_ZERO])

    cols_drop = ['SkinThickness', 'DiabetesPedigreeFunction']
    return dfc.drop(columns=cols_drop).copy()


# ------------------------------------------------------------
# Base 5 — Removendo Variável Insulin (com MICE)
# ------------------------------------------------------------
def base_6_sem_insulin(dfin: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """Imputação MICE + remoção de SkinThickness, DiabetesPedigreeFunction e Insulin."""
    dfc = dfin.copy()
    dfc[COLS_ZERO] = dfc[COLS_ZERO].replace(0, np.nan)

    # Imputação MICE
    imp = IterativeImputer(random_state=random_state, max_iter=10)
    dfc[COLS_ZERO] = imp.fit_transform(dfc[COLS_ZERO])

    cols_drop = ['SkinThickness', 'DiabetesPedigreeFunction', 'Insulin']
    return dfc.drop(columns=cols_drop).copy()


# ============================================================
# CONSTRUÇÃO DAS BASES (apenas 5 modelos solicitados)
# ============================================================
bases = {
    "1_RemovendoZeros":        base_2_removendo_zeros(df),
    "2_MICE_SMOTE_SemInsulin": base_8_mice_smote_sem_insulin(df),
    "3_MICE_SMOTE":            base_4_mice_smote(df),
    "4_SemBaixaImportancia":   base_5_removendo_baixa_importancia(df),
    "5_SemInsulin":            base_6_sem_insulin(df),
}

# Visualização do tamanho de cada base
for nome, b in bases.items():
    print(f"{nome:<25} -> {b.shape[0]} linhas, {b.shape[1]} colunas")


### **3. Análise Descritiva dos Dados**


In [ ]:

# Configurações globais
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

# =======================
# Gráfico de distribuição do Outcome
# =======================
fig, ax = plt.subplots(figsize=(6, 4))
bars = sns.countplot(
    x="Outcome",
    data=df,
    palette=["#3BAEA0", "#F45B69"],  # Cores suaves e contrastantes
    width=0.6,
    ax=ax,
    edgecolor="black"
)

# Adiciona rótulos sobre as barras com caixas coloridas
for p in bars.patches:
    height = p.get_height()
    ax.text(
        p.get_x() + p.get_width()/2,
        height + 10,  # posição acima da barra
        f"{int(height)}",
        ha="center", va="bottom",
        fontsize=10, fontweight="bold",
        color="white",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#3BAEA0" if p.get_x() < 0.5 else "#F45B69", edgecolor="none")
    )

# Título e eixos
ax.set_title("Distribuição do Outcome (0 = Não diabética, 1 = Diabética)", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Classe")
ax.set_ylabel("Número de Registros")

# Remove grades e bordas laterais
sns.despine(left=False, bottom=False)
ax.grid(False)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# TABELA 2 — Estatísticas descritivas
# ============================================================

import pandas as pd
import numpy as np

# Dicionário com os nomes abreviados, como no artigo
colunas = {
    'Pregnancies': 'Preg',
    'Glucose': 'Gluc',
    'BloodPressure': 'BP',
    'SkinThickness': 'Skin',
    'Insulin': 'Insulin',
    'BMI': 'BMI',
    'DiabetesPedigreeFunction': 'DPF',
    'Age': 'Age'
}

# Selecionar as colunas
df_tabela = df[list(colunas.keys())].rename(columns=colunas)


# Calcular estatísticas
tabela_descr = pd.DataFrame({
    'Min.': df_tabela.min(),
    '1st Qu.': df_tabela.quantile(0.25),
    'Median': df_tabela.median(),
    'Mean': df_tabela.mean(),
    '3rd Qu.': df_tabela.quantile(0.75),
    'Max.': df_tabela.max()
}).T.round(2)

# Exibir tabela no formato desejado
display(tabela_descr)

# (opcional) salvar para Word ou Excel
# tabela_descr.to_excel("Tabela2_Estatisticas_Descritivas.xlsx")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# Boxplots das variáveis preditoras – Base Original
# =========================

sns.set_theme(style="white")  # fundo branco, sem grades
cols_plot = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
             'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cols_plot):
    sns.boxplot(
        y=df[col],
        color=sns.color_palette("crest")[3],
        width=0.5,
        linewidth=0.8,
        fliersize=2.5,     # tamanho reduzido dos outliers
        boxprops={'edgecolor': 'black'},
        whiskerprops={'color': 'black'},
        capprops={'color': 'black'},
        medianprops={'color': 'black', 'linewidth': 1.2},
        ax=axes[i]
    )
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")
    axes[i].grid(False)  # remove grades
    sns.despine(ax=axes[i])  # remove bordas sup. e direita

# Remove eixos extras se houver
for j in range(len(cols_plot), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    "Distribuição das Variáveis Preditivas – Boxplots (Base Original)",
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


In [ ]:
# ------------------------------------------------------------
# Histogramas das variáveis preditoras
# ------------------------------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="white")

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

cols_plot = ['Pregnancies', 'Glucose', 'BloodPressure',
             'SkinThickness', 'Insulin', 'BMI',
             'DiabetesPedigreeFunction', 'Age']

for i, col in enumerate(cols_plot):
    sns.histplot(
        df[col],
        bins=20,
        kde=True,
        color=sns.color_palette("crest")[4],
        edgecolor='black',
        linewidth=0.6,
        ax=axes[i]
    )
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequência', fontsize=9)
    axes[i].grid(False)
    sns.despine(ax=axes[i])

fig.delaxes(axes[-1])
fig.suptitle('Distribuições das Variáveis Preditivas ',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


In [ ]:
# =========================
#  Matriz de Correlação
# =========================

# Cálculo da correlação
corr = df.corr(numeric_only=True)

# Estilo visual clean
sns.set_theme(style="white")
plt.rcParams.update({
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

# Figura
fig, ax = plt.subplots(figsize=(8, 6))

# Paleta moderna, fundo branco
sns.heatmap(
    corr,
    cmap="crest",               # Paleta verde-azulada
    annot=True,                 # Mostra os valores numéricos
    fmt=".2f",                  # Duas casas decimais
    linewidths=0.6,             # Linhas divisórias finas
    cbar_kws={"shrink": 0.8, "label": "Correlação de Pearson"},
    square=True,
    annot_kws={"size": 9, "color": "black"},  # Texto legível
    ax=ax
)

# Título e estética geral
ax.set_title("Matriz de Correlação entre as Variáveis (Base Original)", fontsize=13, fontweight="bold", pad=15)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
sns.despine(left=False, bottom=False)
plt.tight_layout()
plt.show()


### **4) Preparação dos dados para modelagem**

In [ ]:
# ========================= PREPARAÇÃO =========================
from typing import Tuple

def split_xy(dfin: pd.DataFrame, target: str = TARGET) -> Tuple[pd.DataFrame, pd.Series]:
    X = dfin.drop(columns=[target]).copy()
    y = dfin[target].copy()
    return X, y

def avaliar_modelo(modelo, X_train, y_train, X_test, y_test, X_full, y_full, nome_base):
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = (modelo.predict_proba(X_test)[:,1]
              if hasattr(modelo, "predict_proba") else None)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    auc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

    cv5 = cross_val_score(modelo, X_full, y_full, cv=5, scoring='accuracy')

    resultados = {
        "Base": nome_base,
        "Acurácia": acc,
        "Precisão": prec,
        "Recall": rec,
        "Especificidade": spec,
        "F1": f1,
        "AUC": auc,
        "CV5_Média": cv5.mean(),
        "CV5_DP": cv5.std()
    }
    return resultados, cm, y_pred

def print_relatorio(nome_base, cm, y_test, y_pred):
    print(f"\n=== {nome_base} ===")
    print("Matriz de confusão:\n", cm)
    print("\nRelatório de classificação:\n",
          classification_report(y_test, y_pred, target_names=['Não diabético','Diabético'],
                                zero_division=0))

### **5) Treinamento e ajuste de hiperparâmetros**

In [ ]:
# ============================================================
# TRABALHO 2 — MODELO RANDOM TREE (otimizado)
# ============================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, train_test_split, RepeatedStratifiedKFold
from scipy.stats import randint, uniform

# ========================= ESCOLHA DA BASE PARA O MODELO =========================
df_treated = bases["8_MICE_SMOTE_SemInsulin"]

# ========================= SEPARAÇÃO ENTRE TREINO E TESTE =========================
X = df_treated.drop(columns=[TARGET])
y = df_treated[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Tamanho dos conjuntos:")
print("Treino:", X_train.shape, " | Teste:", X_test.shape)

# ========================= CONFIGURAÇÃO DE VALIDAÇÃO =========================
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

# ========================= ESPAÇO DE BUSCA AMPLIADO =========================
param_dist = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': randint(3, 25),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 8),
    'ccp_alpha': uniform(0.0, 0.005)
}

# ========================= MODELO BASE (Random Tree) =========================
rt = DecisionTreeClassifier(
    splitter='random',       # divisões aleatórias
    max_features='sqrt',     # subconjunto aleatório de atributos
    random_state=42
)

# ========================= RANDOMIZED SEARCH AVANÇADO =========================
random_search = RandomizedSearchCV(
    estimator=rt,
    param_distributions=param_dist,
    n_iter=150,              # mais combinações
    scoring='roc_auc',       # foco em discriminação global
    cv=cv,                   # validação mais robusta
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Treinamento
random_search.fit(X_train, y_train)

# Melhor modelo encontrado
best_rt = random_search.best_estimator_

print("\n=== Melhores hiperparâmetros encontrados (Random Tree) ===")
print(random_search.best_params_)
print("Média de AUC (CV=10x3):", round(random_search.best_score_, 4))


In [ ]:
# ========================= TREINAMENTO - Melhores hiperparâmetros (Random Tree) =========================
params_fixos = {
    'criterion': 'entropy',
    'splitter': 'random',
    'max_features': 'sqrt',
    'max_depth': 12,
    'min_samples_split': 18,
    'min_samples_leaf': 3,
    'ccp_alpha': 0.0036306042203592404,
    'random_state': RANDOM_STATE
}

### **6) Avaliação e validação do modelo**

In [ ]:
# ========================= 6. AVALIAÇÃO (8 CENÁRIOS) — RANDOM TREE =========================
resultados = []
conf_mats = {}
predicoes = {}

# Definir parâmetros fixos do modelo (pode usar o resultado do RandomizedSearch anterior)
params_fixos = {
    'criterion': 'entropy',
    'splitter': 'random',      # aleatoriedade nas divisões
    'max_features': 'sqrt',    # subconjunto aleatório de atributos por nó
    'max_depth': 12,           # exemplo — substitua pelo melhor valor encontrado
    'min_samples_split': 18,
    'min_samples_leaf': 2,
    'ccp_alpha': 0.0036306042203592404,
    'random_state': 42
}

for nome_base, dbase in bases.items():
    dbase = dbase.dropna(subset=[TARGET]).copy()  # segurança

    X, y = split_xy(dbase, target=TARGET)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    # Modelo Random Tree com os parâmetros definidos acima
    modelo = DecisionTreeClassifier(**params_fixos)

    res, cm, y_pred = avaliar_modelo(
        modelo, X_train, y_train, X_test, y_test, X, y, nome_base
    )
    resultados.append(res)
    conf_mats[nome_base] = cm
    predicoes[nome_base] = (y_test, y_pred)

    # Relatório compacto
    print(f"\n===== {nome_base} (Random Tree) =====")
    print_relatorio(nome_base, cm, y_test, y_pred)

# ============================================================
# Tabela consolidada de resultados
# ============================================================
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados[
    ["Base","Acurácia","Precisão","Recall","Especificidade","F1","AUC","CV5_Média","CV5_DP"]
].sort_values(by=["AUC","F1","Acurácia"], ascending=False)

print("\n=== Tabela Comparativa (ordenada por AUC, F1, Acurácia) ===")
display(df_resultados)


### **7) Interpretação e análise dos resultados**

In [ ]:
# ========================= 7. INTERPRETAÇÃO — RANDOM TREE =========================
# Escolha por maior AUC; em caso de empate, F1 e depois Acurácia
ord_cols = ["CV5_Média","F1","Acurácia"]
melhor_idx = df_resultados.sort_values(by=ord_cols, ascending=False).index[0]
melhor_base = df_resultados.loc[melhor_idx, "Base"]
print("Melhor base por critérios (CV5_Média > F1 > Acurácia):", melhor_base)

# Reconstruir splits dessa base
dbase = bases[melhor_base].dropna(subset=[TARGET]).copy()
X, y = split_xy(dbase, target=TARGET)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Modelo final — Random Tree (usando os parâmetros ótimos)
modelo_final = DecisionTreeClassifier(
    criterion='gini',
    splitter='random',       # aleatoriedade nas divisões
    max_features='sqrt',     # subconjunto aleatório de atributos
    max_depth=10,            # exemplo — substitua pelo melhor valor do RandomSearch
    min_samples_split=5,
    min_samples_leaf=2,
    ccp_alpha=0.001,
    random_state=42
)

# Treinamento final
modelo_final.fit(X_train, y_train)
y_pred = modelo_final.predict(X_test)

# ========================= MATRIZ DE CONFUSÃO =========================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cbar=False,
            xticklabels=['Pred: Não diabético','Pred: Diabético'],
            yticklabels=['Real: Não diabético','Real: Diabético'])
plt.title(f'Matriz de Confusão — {melhor_base} (Random Tree)')
plt.xlabel('Classe Predita'); plt.ylabel('Classe Real')
plt.tight_layout(); plt.show()

# ========================= IMPORTÂNCIA DAS VARIÁVEIS =========================
importances = pd.Series(modelo_final.feature_importances_, index=X.columns).sort_values(ascending=False)
ax = importances.plot(kind='bar', figsize=(10,4))
ax.set_title(f'Importância das Variáveis — {melhor_base} (Random Tree)')
ax.set_ylabel('Importância relativa')
plt.tight_layout(); plt.show()

# ========================= ÁRVORE (visualização resumida) =========================
plt.figure(figsize=(18, 9))
plot_tree(
    modelo_final,
    feature_names=X.columns,
    class_names=['Não diabético','Diabético'],
    filled=True,
    rounded=True,
    fontsize=10,
    #max_depth=3    # limite para deixar legível
)
plt.title(f"Estrutura da Random Tree — {melhor_base} (até 3 níveis)")
plt.show()


Random Forest - Aplicando

In [ ]:
# Validação cruzada mais leve
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=1, random_state=42)

# Espaço de busca reduzido
param_dist = {
    'n_estimators': randint(100, 300),
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': randint(5, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 8),
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True],
    'ccp_alpha': uniform(0.0, 0.003)
}

random_search = RandomizedSearchCV(
    estimator=rt,
    param_distributions=param_dist,
    n_iter=80,              # menos combinações
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)



In [ ]:
# ========================= 6. AVALIAÇÃO (8 CENÁRIOS) — RANDOM FOREST =========================
from sklearn.ensemble import RandomForestClassifier

resultados = []
conf_mats = {}
predicoes = {}

# Parâmetros fixos — substituindo pelos melhores obtidos no RandomizedSearchCV
params_fixos = {
    'n_estimators': 200,          # número de árvores
    'criterion': 'entropy',       # função de impureza
    'max_depth': 7,              # profundidade máxima
    'min_samples_split': 18,      # mínimo para dividir
    'min_samples_leaf': 3,        # mínimo em cada folha
    'max_features': 'sqrt',       # subconjunto aleatório de variáveis
    'bootstrap': True,            # amostragem com reposição
    'ccp_alpha': 0.00363,         # poda por complexidade
    'random_state': RANDOM_STATE
}


for nome_base, dbase in bases.items():
    dbase = dbase.dropna(subset=[TARGET]).copy()  # segurança

    X, y = split_xy(dbase, target=TARGET)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    # Modelo Random Forest com os parâmetros definidos acima
    modelo = RandomForestClassifier(**params_fixos)

    res, cm, y_pred = avaliar_modelo(
        modelo, X_train, y_train, X_test, y_test, X, y, nome_base
    )
    resultados.append(res)
    conf_mats[nome_base] = cm
    predicoes[nome_base] = (y_test, y_pred)

    # Relatório compacto
    print(f"\n===== {nome_base} (Random Forest) =====")
    print_relatorio(nome_base, cm, y_test, y_pred)

# ============================================================
# Tabela consolidada de resultados
# ============================================================
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados[
    ["Base","Acurácia","Precisão","Recall","Especificidade","F1","AUC","CV5_Média","CV5_DP"]
].sort_values(by=["AUC","F1","Acurácia"], ascending=False)

print("\n=== Tabela Comparativa (ordenada por AUC, F1, Acurácia) ===")
display(df_resultados)


In [ ]:
# ========================= 7. INTERPRETAÇÃO — RANDOM FOREST =========================
# Escolha por maior AUC; em caso de empate, F1 e depois Acurácia
ord_cols = ["CV5_Média","F1","Acurácia"]
melhor_idx = df_resultados.sort_values(by=ord_cols, ascending=False).index[0]
melhor_base = df_resultados.loc[melhor_idx, "Base"]
print("Melhor base por critérios (CV5_Média > F1 > Acurácia):", melhor_base)

# Reconstruir splits dessa base
dbase = bases[melhor_base].dropna(subset=[TARGET]).copy()
X, y = split_xy(dbase, target=TARGET)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)




# Modelo final — Random Forest (usando os parâmetros ótimos)
modelo_final = RandomForestClassifier(
    n_estimators=200,
    criterion='entropy',
    max_depth=7,
    min_samples_split=18,
    min_samples_leaf=3,
    max_features='sqrt',
    bootstrap=True,
    ccp_alpha=0.00363,
    random_state=RANDOM_STATE
)

# Treinamento final
modelo_final.fit(X_train, y_train)
y_pred = modelo_final.predict(X_test)

# ========================= MATRIZ DE CONFUSÃO =========================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cbar=False,
            xticklabels=['Pred: Não diabético','Pred: Diabético'],
            yticklabels=['Real: Não diabético','Real: Diabético'])
plt.title(f'Matriz de Confusão — {melhor_base} (Random Forest)')
plt.xlabel('Classe Predita'); plt.ylabel('Classe Real')
plt.tight_layout(); plt.show()

# ========================= IMPORTÂNCIA DAS VARIÁVEIS =========================
importances = pd.Series(modelo_final.feature_importances_, index=X.columns).sort_values(ascending=False)
ax = importances.plot(kind='bar', figsize=(10,4))
ax.set_title(f'Importância das Variáveis — {melhor_base} (Random Forest)')
ax.set_ylabel('Importância relativa')
plt.tight_layout(); plt.show()

# ========================= VISUALIZAÇÃO DE UMA ÁRVORE INTERNA (opcional) =========================
# Visualiza apenas uma árvore aleatória do ensemble para interpretação
plt.figure(figsize=(18, 9))
from sklearn.tree import plot_tree

# Selecionar uma árvore individual do modelo
arvore_exemplo = modelo_final.estimators_[0]

plot_tree(
    arvore_exemplo,
    feature_names=X.columns,
    class_names=['Não diabético','Diabético'],
    filled=True,
    rounded=True,
    fontsize=9,
    #max_depth=3   # limite para legibilidade
)
plt.title(f"Estrutura de uma Árvore Interna — {melhor_base} (Random Forest, até 3 níveis)")
plt.show()

# ========================= MÉTRICAS RESUMIDAS =========================
from sklearn.metrics import classification_report, roc_auc_score

auc = roc_auc_score(y_test, modelo_final.predict_proba(X_test)[:,1])
print("\nAUC:", round(auc, 4))
print("\nRelatório de classificação:\n",
      classification_report(y_test, y_pred, target_names=['Não diabético','Diabético'], zero_division=0))


In [ ]:
# ========================= Importância das Variáveis — Barras Horizontais com Rótulos =========================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Série de importância (ordem decrescente)
importances = (
    pd.Series(modelo_final.feature_importances_, index=X.columns)
    .sort_values(ascending=True)  # ascending=True para o menor embaixo e o maior no topo
)

# Estilo visual
sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(
    importances.index,
    importances.values,
    color="#65A68C",       # verde-oliva
    #edgecolor="black",
    height=0.6
)

# Adiciona rótulos nas barras
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 0.01,                   # deslocamento à direita da barra
        bar.get_y() + bar.get_height()/2,
        f"{width:.2f}",
        ha="left", va="center",
        fontsize=9, fontweight="semibold", color="black"
    )

# Estética geral
ax.set_title(f"Importância das Variáveis — {melhor_base}", fontsize=13, pad=15)
ax.set_xlabel("Importância relativa", fontsize=11)
ax.set_ylabel("")
ax.set_xlim(0, importances.max() + 0.1)
ax.grid(False)
sns.despine()

plt.tight_layout()
plt.show()


In [ ]:
# ========================= 7. INTERPRETAÇÃO — RANDOM FOREST =========================
# Escolher manualmente a base desejada
melhor_base = "2_MICE_SMOTE_SemInsulin"
print(f"Rodando análise final com a base: {melhor_base}")

# Reconstruir splits dessa base
dbase = bases[melhor_base].dropna(subset=[TARGET]).copy()
X, y = split_xy(dbase, target=TARGET)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Modelo final — Random Forest (usando os parâmetros ótimos)
modelo_finalrd = RandomForestClassifier(
    n_estimators=200,
    criterion='entropy',
    max_depth=7,
    min_samples_split=18,
    min_samples_leaf=3,
    max_features='sqrt',
    bootstrap=True,
    ccp_alpha=0.00363,
    random_state=RANDOM_STATE
)

# Treinamento final
modelo_finalrd.fit(X_train, y_train)
y_pred = modelo_finalrd.predict(X_test)

# ========================= MATRIZ DE CONFUSÃO =========================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cbar=False,
            xticklabels=['Pred: Não diabético','Pred: Diabético'],
            yticklabels=['Real: Não diabético','Real: Diabético'])
plt.title(f'Matriz de Confusão — {melhor_base} (Random Forest)')
plt.xlabel('Classe Predita'); plt.ylabel('Classe Real')
plt.tight_layout(); plt.show()

# ========================= IMPORTÂNCIA DAS VARIÁVEIS =========================
importances = pd.Series(modelo_finalrd.feature_importances_, index=X.columns).sort_values(ascending=False)
ax = importances.plot(kind='bar', figsize=(10,4))
ax.set_title(f'Importância das Variáveis — {melhor_base} (Random Forest)')
ax.set_ylabel('Importância relativa')
plt.tight_layout(); plt.show()

# ========================= VISUALIZAÇÃO DE UMA ÁRVORE INTERNA (opcional) =========================
plt.figure(figsize=(18, 9))
from sklearn.tree import plot_tree

arvore_exemplo = modelo_finalrd.estimators_[0]

plot_tree(
    arvore_exemplo,
    feature_names=X.columns,
    class_names=['Não diabético','Diabético'],
    filled=True,
    rounded=True,
    fontsize=9,
    #max_depth=3
)
plt.title(f"Estrutura de uma Árvore Interna — {melhor_base} (Random Forest, até 3 níveis)")
plt.show()

# ========================= MÉTRICAS RESUMIDAS =========================
from sklearn.metrics import classification_report, roc_auc_score

auc = roc_auc_score(y_test, modelo_finalrd.predict_proba(X_test)[:,1])
print("\nAUC:", round(auc, 4))
print("\nRelatório de classificação:\n",
      classification_report(y_test, y_pred, target_names=['Não diabético','Diabético'], zero_division=0))


In [ ]:
# ========================= Importância das Variáveis — Barras Horizontais com Rótulos =========================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Série de importância (ordem decrescente)
importances = (
   pd.Series(modelo_finalrd.feature_importances_, index=X_train.columns)
    .sort_values(ascending=True)  # menor embaixo, maior no topo
)

# Estilo visual
sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(
    importances.index,
    importances.values,
    color="#4C9A84",       # verde mais escuro, tom institucional
    height=0.6
)

# Adiciona rótulos numéricos nas barras
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 0.01,                      # deslocamento à direita da barra
        bar.get_y() + bar.get_height() / 2,
        f"{width:.3f}",                    # 3 casas decimais
        ha="left", va="center",
        fontsize=9, fontweight="semibold", color="black"
    )

# Estética geral
ax.set_title(f"Importância das Variáveis — {melhor_base} (Random Forest)", fontsize=13, pad=15)
ax.set_xlabel("Importância relativa", fontsize=11)
ax.set_ylabel("")
ax.set_xlim(0, importances.max() + 0.05)
ax.grid(False)
sns.despine()

plt.tight_layout()
plt.show()


In [ ]:
# ========================= 7. INTERPRETAÇÃO — RANDOM FOREST (Sem 3 últimas variáveis) =========================

melhor_base = "2_MICE_SMOTE_SemInsulin"
print(f"Rodando análise final com a base: {melhor_base}")

# --- Reconstruir a base e remover as 3 últimas variáveis
dbase = bases[melhor_base].dropna(subset=[TARGET]).copy()
X, y = split_xy(dbase, target=TARGET)

# Remove as 3 últimas variáveis
variaveis_remover = ['DiabetesPedigreeFunction', 'BloodPressure', 'Pregnancies']
X = X.drop(columns=variaveis_remover)

# --- Separação treino/teste
from sklearn.model_selection import train_test_split, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# --- Modelo final Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

modelo_rf_reduzido = RandomForestClassifier(
    n_estimators=200,
    criterion='entropy',
    max_depth=7,
    min_samples_split=18,
    min_samples_leaf=3,
    max_features='sqrt',
    bootstrap=True,
    ccp_alpha=0.00363,
    random_state=RANDOM_STATE
)

# --- Treinamento
modelo_rf_reduzido.fit(X_train, y_train)
y_pred = modelo_rf_reduzido.predict(X_test)

# --- Validação cruzada 5-fold
cv5_scores = cross_val_score(modelo_rf_reduzido, X, y, cv=5, scoring='accuracy')
cv5_media = cv5_scores.mean()
cv5_dp = cv5_scores.std()

print(f"\nValidação Cruzada 5-Fold:")
print(f"Acurácia média: {cv5_media:.3f}")
print(f"Desvio padrão:  {cv5_dp:.3f}")

# --- AUC e métricas
auc = roc_auc_score(y_test, modelo_rf_reduzido.predict_proba(X_test)[:,1])
print(f"\nAUC: {auc:.3f}\n")
print("Relatório de classificação:\n")
print(classification_report(y_test, y_pred, target_names=['Não diabético', 'Diabético'], zero_division=0))

# --- Matriz de Confusão
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cbar=False,
            xticklabels=['Pred: Não diabético','Pred: Diabético'],
            yticklabels=['Real: Não diabético','Real: Diabético'])
plt.title(f'Matriz de Confusão — {melhor_base} (Random Forest - sem 3 variáveis)')
plt.xlabel('Classe Predita')
plt.ylabel('Classe Real')
plt.tight_layout()
plt.show()

# --- Importância das Variáveis
importances = (
    pd.Series(modelo_rf_reduzido.feature_importances_, index=X_train.columns)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(importances.index, importances.values, color="#4C9A84", height=0.6)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2,
            f"{width:.3f}", ha="left", va="center",
            fontsize=9, fontweight="semibold", color="black")

ax.set_title(f"Importância das Variáveis — {melhor_base} (Random Forest - sem 3 variáveis)", fontsize=13, pad=15)
ax.set_xlabel("Importância relativa", fontsize=11)
ax.set_xlim(0, importances.max() + 0.05)
sns.despine()
plt.tight_layout()
plt.show()


## Interpretação técnica e limitações

O estudo compara cenários de tratamento de dados da base Pima Indians Diabetes por árvores aleatórias e Random Forest. A base inicial apresentada contém 768 registros e 9 variáveis, incluindo a variável resposta.

**Resultados registrados:** a seleção por critérios de validação cruzada identifica o cenário `3_MICE_SMOTE` em partes da análise. Há avaliações adicionais com a base `2_MICE_SMOTE_SemInsulin`, nas quais o notebook imprime AUC de 0,874 e, em uma configuração com exclusão de três variáveis, AUC de 0,865 e acurácia média de validação cruzada de 0,793 (desvio padrão de 0,047). Essas métricas se referem a configurações distintas e não devem ser comparadas como se fossem o mesmo experimento.

**Pontos de atenção:** os procedimentos de imputação, seleção de atributos e SMOTE devem ocorrer dentro de cada partição de treinamento para evitar vazamento de informação. A escolha final deve priorizar uma métrica predefinida, considerar recall da classe positiva e registrar intervalos de incerteza. O termo “Random Tree” é mantido como denominação do exercício, mas recomenda-se especificar a classe efetivamente utilizada no código. Estas observações interpretam saídas gravadas no notebook, sem reexecução.